<a href="https://colab.research.google.com/github/Trickwillfrit/EPFL_Rolex/blob/main/notebooks/evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluation of recommendation models

In this notebook, we build the evaluation pipeline for the recommendation system.

Our goals are:
- create a validation setup from the training interactions
- define the ground-truth items for each user
- implement Precision@10 and Recall@10
- compare recommendation models fairly

In [76]:
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = "https://raw.githubusercontent.com/Trickwillfrit/EPFL_Rolex/main/"

interactions = pd.read_csv(BASE_URL + "interactions_train.csv")
items = pd.read_csv(BASE_URL + "items.csv")
submission = pd.read_csv(BASE_URL + "sample_submission.csv")

print("Interactions shape:", interactions.shape)
display(interactions.head())

Interactions shape: (87047, 3)


,u,i,t
0,4456,8581,1.687541e+09
1,142,1964,1.679585e+09
2,362,3705,1.706872e+09
3,1809,11317,1.673533e+09
4,4384,1323,1.681402e+09


In [77]:
print("Interactions shape:", interactions.shape)
display(interactions.head())
print(interactions.columns)

Interactions shape: (87047, 3)


,u,i,t
0,4456,8581,1.687541e+09
1,142,1964,1.679585e+09
2,362,3705,1.706872e+09
3,1809,11317,1.673533e+09
4,4384,1323,1.681402e+09


Index(['u', 'i', 't'], dtype='object')


In [78]:
# Convert the timestamp column into a readable datetime format
interactions["datetime"] = pd.to_datetime(interactions["t"], unit="s")

# Sort interactions by user and time
interactions = interactions.sort_values(["u", "datetime"]).reset_index(drop=True)

# Quick check
display(interactions.head())

,u,i,t,datetime
0,0,0,1.680191e+09,2023-03-30 15:44:30
1,0,1,1.680783e+09,2023-04-06 12:13:54
2,0,2,1.680801e+09,2023-04-06 17:15:08
3,0,3,1.683715e+09,2023-05-10 10:35:45
4,0,3,1.683715e+09,2023-05-10 10:35:50


In [79]:
# Count the number of interactions for each user
user_counts = interactions["u"].value_counts()

# Display summary statistics
print(user_counts.describe())

count    7838.000000
mean       11.105767
std        16.441875
min         3.000000
25%         3.000000
50%         6.000000
75%        11.000000
max       385.000000
Name: count, dtype: float64


## Validation strategy

To evaluate our recommendation models, we use a leave-one-out strategy.

For each user, we keep the **last interaction in time** as the validation target, and we use all previous interactions as the training history.

This setup is appropriate for recommendation because it simulates a realistic situation: we try to predict a future item from the user's past behavior.

## What is the validation
The validation set is the part of the data that we keep aside to evaluate the model.

The model is trained only on the training set, and then we test whether it can correctly recommend the items contained in the validation set.

In [80]:
# For each user, keep the last interaction as validation
valid = interactions.groupby("u").tail(1).copy()

# Keep all previous interactions as training
train = interactions.drop(valid.index).copy()

# Reset indices for cleaner tables
train = train.reset_index(drop=True)
valid = valid.reset_index(drop=True)

# Check the sizes of the two splits
print("Train shape:", train.shape)
print("Validation shape:", valid.shape)

# Show the first rows
display(train.head())
display(valid.head())

Train shape: (79209, 4)
Validation shape: (7838, 4)


,u,i,t,datetime
0,0,0,1.680191e+09,2023-03-30 15:44:30
1,0,1,1.680783e+09,2023-04-06 12:13:54
2,0,2,1.680801e+09,2023-04-06 17:15:08
3,0,3,1.683715e+09,2023-05-10 10:35:45
4,0,3,1.683715e+09,2023-05-10 10:35:50


,u,i,t,datetime
0,0,24,1.713272e+09,2024-04-16 12:58:06
1,1,39,1.707759e+09,2024-02-12 17:37:24
2,2,92,1.709921e+09,2024-03-08 17:59:25
3,3,172,1.714129e+09,2024-04-26 11:01:00
4,4,195,1.714578e+09,2024-05-01 15:35:37


In [81]:
# Build the ground-truth dictionary from the validation set
# Each user is associated with the item we want the model to recover
ground_truth = valid.groupby("u")["i"].apply(list).to_dict()

# Build the training history dictionary
# For each user, store the set of items seen in training
train_user_items = train.groupby("u")["i"].apply(set).to_dict()

# Quick checks
print("Number of users in ground truth:", len(ground_truth))
print("Number of users in train history:", len(train_user_items))

# Show one example
example_user = list(ground_truth.keys())[0]
print("Example user:", example_user)
print("Train items:", train_user_items[example_user])
print("Validation item:", ground_truth[example_user])

Number of users in ground truth: 7838
Number of users in train history: 7838
Example user: 0
Train items: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23}
Validation item: [24]


In [82]:
# Function to compute Precision@K for one user
def precision_at_k(recommended_items, true_items, k=10): #among the top 10 books recommended by the model, how many are actually correct?
    # Keep only the first k recommended items
    recommended_items = recommended_items[:k]

    # Count how many recommended items are actually relevant
    hits = len(set(recommended_items) & set(true_items))

    # Precision@K = number of relevant recommended items / k
    return hits / k

In [83]:
# Function to compute Recall@K for one user
def recall_at_k(recommended_items, true_items, k=10):
    # Keep only the first k recommended items
    recommended_items = recommended_items[:k]

    # Count how many recommended items are actually relevant
    hits = len(set(recommended_items) & set(true_items))

    # Recall@K = number of relevant recommended items / number of true relevant items
    return hits / len(true_items)

In [84]:
# Function to evaluate a recommender over all users
def evaluate_recommender(recommendations, ground_truth, k=10):
    precision_scores = []
    recall_scores = []

    # Loop over all users in the ground truth
    for user, true_items in ground_truth.items():
        # Get the recommended items for this user
        recommended_items = recommendations.get(user, [])

        # Compute Precision@K and Recall@K for this user
        precision_scores.append(precision_at_k(recommended_items, true_items, k))
        recall_scores.append(recall_at_k(recommended_items, true_items, k))

    # Return the average scores across users
    return {
        "Precision@K": sum(precision_scores) / len(precision_scores),
        "Recall@K": sum(recall_scores) / len(recall_scores)
    }

In [85]:
# Create a small fake recommendation dictionary for a few users
fake_recommendations = {
    0: [5, 8, 24, 31, 44, 10, 3, 2, 90, 11],   # contains the true item 24
    1: [7, 12, 19, 25, 39, 41, 50, 60, 70, 80], # contains the true item 39
    2: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]          # does not contain the true item 92
}

# Create the matching fake ground truth
fake_ground_truth = {
    0: [24],
    1: [39],
    2: [92]
}

# Evaluate the fake recommender
fake_results = evaluate_recommender(fake_recommendations, fake_ground_truth, k=10)
print(fake_results)

{'Precision@K': 0.06666666666666667, 'Recall@K': 0.6666666666666666}


## Understanding Precision@10 and Recall@10

In this notebook, each user has **one validation item**, which is the last book they interacted with.

When we say that a model **recovers** a book, we simply mean that the true validation book appears somewhere in the top-10 recommendation list.

### Precision@10
Precision@10 answers the question:

**Among the 10 books recommended by the model, how many are correct?**

Since we always recommend 10 books, the denominator is 10.

For example, if the true validation book is in the recommendation list, then there is 1 correct item out of 10:

Precision@10 = 1 / 10 = 0.1

If the true book is not in the list, then:

Precision@10 = 0 / 10 = 0

### Recall@10
Recall@10 answers the question:

**Among the true relevant books, how many did the model recover?**

In our current setup, each user has only **one true validation book**, so the denominator is 1.

If the model recommends that true book, then:

Recall@10 = 1 / 1 = 1

If the model does not recommend it, then:

Recall@10 = 0 / 1 = 0

### Why the denominators are different
- Precision uses the number of recommended items, because it measures the quality of the recommendation list.
- Recall uses the number of true relevant items, because it measures whether the model managed to recover the relevant items.

So in our leave-one-out setup:
- if the true book is in the top 10, then Precision@10 = 0.1 and Recall@10 = 1
- otherwise, both metrics are 0

In [86]:
# Count item popularity in the training set
item_popularity = train["i"].value_counts()

# Keep the item IDs ordered by popularity
popular_items = item_popularity.index.tolist()

# Build recommendations for each user:
# recommend the most popular items that the user has not already seen
popular_recommendations = {}

for user in ground_truth.keys():
    seen_items = train_user_items[user]

    # Filter out items already seen by the user
    recs = [item for item in popular_items if item not in seen_items][:10]

    popular_recommendations[user] = recs

# Show recommendations for one example user
example_user = list(popular_recommendations.keys())[0]
print("Example user:", example_user)
print("Recommended items:", popular_recommendations[example_user])
print("True validation item:", ground_truth[example_user])

Example user: 0
Recommended items: [3055, 11366, 10715, 8999, 611, 4426, 53, 2820, 14555, 14578]
True validation item: [24]


In [87]:
# Show the metadata of item 24
display(items[items["i"] == 24])

,Title,Author,ISBN Valid,Publisher,Subjects,i
24,Le sentiment de justice dans les relations soc...,"Kellerhals, Jean, 1941-",2130486851; 9782130486855,Presses universitaires de France,justice--relations sociales; justice distribut...,24


## Interpreting one recommendation example

This output shows the recommendations for one example user.

- `Recommended items` is the list of 10 books proposed by the popularity baseline.
- `True validation item` is the real last book the user interacted with in the dataset.

We compare the two lists to check whether the true validation item appears in the top-10 recommendations. In this example, it does not, so the model fails for this user.

In [88]:
# Evaluate the popularity baseline on all users
popular_results = evaluate_recommender(popular_recommendations, ground_truth, k=10)

print("Popularity baseline results:")
print(popular_results)

Popularity baseline results:
{'Precision@K': 0.0004082674151569278, 'Recall@K': 0.004082674151569278}


## Popularity baseline results

The popularity baseline obtains very low scores on the validation set.

This means that recommending the same globally popular books to every user is not sufficient to recover the users' true next interactions. In other words, the recommendation task requires more personalization than a simple popularity-based approach can provide.

This baseline is still useful because it gives us a reference point: more advanced models should perform better than this.

In [89]:
# Count how many users have their true validation item in the top-10 recommendations
num_hits = 0

for user, true_items in ground_truth.items():
    recommended_items = popular_recommendations[user][:10]
    if len(set(recommended_items) & set(true_items)) > 0:
        num_hits += 1

# Print the number and percentage of successful recommendations
print("Number of users with a hit in top 10 on 7838 users:", num_hits)
print("Percentage of users with a hit in top 10:", round(100 * num_hits / len(ground_truth), 3), "%")

Number of users with a hit in top 10 on 7838 users: 32
Percentage of users with a hit in top 10: 0.408 %


## User-user collaborative filtering

After the popularity baseline, we now test a personalized recommendation approach: user-user collaborative filtering.

The idea is to recommend books based on the behavior of similar users. If two users interacted with similar books in the past, then books read by one of them may be relevant for the other.

To do this, we first build an inverted index that tells us which users interacted with each item.

In [90]:
# Build a dictionary: for each item, store the set of users who interacted with it
item_users = train.groupby("i")["u"].apply(set).to_dict()

# Quick check
print("Number of items in item_users:", len(item_users))

example_item = list(item_users.keys())[0]
print("Example item:", example_item)
print("Users who interacted with this item:", item_users[example_item])

Number of items in item_users: 14979
Example item: 0
Users who interacted with this item: {0, 224, 6018}


## Inverted index for user-user collaborative filtering

This step builds an inverted index from the training data.

Instead of storing, for each user, the set of books they interacted with, we store for each item the set of users who interacted with it.

For example, if `item_users[0] = {0, 224, 6018}`, this means that item 0 was interacted with by users 0, 224, and 6018 in the training set.

This structure is useful because it helps identify users who share common items and may therefore have similar preferences.

In [91]:
# Function to compute similarity between two users using Jaccard similarity
def user_similarity(user_a, user_b, train_user_items):
    items_a = train_user_items.get(user_a, set())
    items_b = train_user_items.get(user_b, set())

    intersection = len(items_a & items_b)
    union = len(items_a | items_b)

    if union == 0:
        return 0

    return intersection / union

## User similarity with Jaccard index

To compare two users, we look at the sets of books they interacted with in the training data.

We use the **Jaccard similarity**, defined as:

- the number of books the two users have in common
- divided by the total number of distinct books seen by at least one of them

So:
- similarity = **1** if the two users interacted with exactly the same books
- similarity = **0** if they have no common books
- similarity is between 0 and 1 otherwise

This gives a simple way to measure whether two users have similar reading histories, which is the basis of user-user collaborative filtering.

In [92]:
# Test the user similarity function on two example users
user_a = 0
user_b = 224

sim_ab = user_similarity(user_a, user_b, train_user_items)

print("User A:", user_a)
print("Items of user A:", train_user_items[user_a])
print()
print("User B:", user_b)
print("Items of user B:", train_user_items[user_b])
print()
print("Jaccard similarity:", sim_ab)

User A: 0
Items of user A: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23}

User B: 224
Items of user B: {0, 2945, 2946, 2947, 2948, 2949, 2950, 2951, 2316, 1328, 1395, 1879, 2199, 61, 1374}

Jaccard similarity: 0.02631578947368421


In [93]:
# Function to find candidate similar users for a target user
def get_candidate_users(target_user, train_user_items, item_users):
    candidate_users = set()

    # Look at all items seen by the target user
    for item in train_user_items.get(target_user, set()):
        # Add all users who also interacted with this item
        candidate_users.update(item_users.get(item, set()))

    # Remove the target user from the candidate set
    candidate_users.discard(target_user)

    return candidate_users

## Candidate users for user-user collaborative filtering

To recommend books for a target user, we do not need to compare that user with every other user in the dataset.

Instead, we first build a set of **candidate users**: users who have interacted with at least one of the same books as the target user.

This is useful because users with no common items automatically have similarity 0, so they are not informative. By restricting the comparison to candidate users, we make the method more efficient and more focused.

In [94]:
# Test the candidate-user function on one example user
target_user = 0
candidate_users = get_candidate_users(target_user, train_user_items, item_users)

print("Target user:", target_user)
print("Number of candidate users:", len(candidate_users))
print("Some candidate users:", list(candidate_users)[:10])

Target user: 0
Number of candidate users: 103
Some candidate users: [769, 1538, 1539, 1284, 770, 5903, 1808, 274, 6418, 4118]


## Meaning of the main variables

- `target_user`: the user for whom we want to find similar users
- `train_user_items`: dictionary that maps each user to the set of items they interacted with in training
- `item_users`: dictionary that maps each item to the set of users who interacted with it
- `candidate_users`: set of users who share at least one item with the target user

This test cell is only used to verify that the candidate-user function works correctly on one example user.

## Candidate users for one example user

For the example user 0, we found 103 candidate users. These are users who share at least one training item with user 0.

This does not yet mean that they are strongly similar, only that they have some overlap. In the next step, we compute similarity scores to identify which of these candidate users are the most relevant neighbors.

In [95]:
# Function to find the top-N most similar users to a target user
def get_top_neighbors(target_user, train_user_items, item_users, top_n=10):
    candidate_users = get_candidate_users(target_user, train_user_items, item_users)

    similarities = []

    # Compute similarity between the target user and each candidate user
    for other_user in candidate_users:
        sim = user_similarity(target_user, other_user, train_user_items)
        similarities.append((other_user, sim))

    # Sort by similarity in descending order
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Keep only the top-N neighbors
    return similarities[:top_n]

## User-user collaborative filtering evaluation

We now apply the user-user collaborative filtering method to all users in the validation set and evaluate its performance with Precision@10 and Recall@10.

In [96]:
# Function to recommend items for one user using user-user collaborative filtering
def recommend_user_user(target_user, train_user_items, item_users, top_n_neighbors=10, k=10):
    # Find the most similar users
    neighbors = get_top_neighbors(target_user, train_user_items, item_users, top_n=top_n_neighbors)

    # Get the items already seen by the target user
    seen_items = train_user_items.get(target_user, set())

    # Store recommendation scores for candidate items
    item_scores = {}

    # Go through each neighbor
    for neighbor_user, similarity in neighbors:
        neighbor_items = train_user_items.get(neighbor_user, set())

        # Add unseen neighbor items to the score dictionary
        for item in neighbor_items:
            if item not in seen_items:
                item_scores[item] = item_scores.get(item, 0) + similarity

    # Sort candidate items by score in descending order
    ranked_items = sorted(item_scores.items(), key=lambda x: x[1], reverse=True)

    # Keep only the top-k item IDs
    recommended_items = [item for item, score in ranked_items[:k]]

    return recommended_items

In [97]:
# Test the recommendation function on one example user
target_user = 0
test_recommendations = recommend_user_user(
    target_user=target_user,
    train_user_items=train_user_items,
    item_users=item_users,
    top_n_neighbors=10,
    k=10
)

print("Target user:", target_user)
print("Recommended items:", test_recommendations)
print("True validation item:", ground_truth[target_user])

Target user: 0
Recommended items: [13307, 9007, 759, 8404, 8096, 13382, 12049, 4565, 8095, 9759]
True validation item: [24]


In [98]:
# Build user-user recommendations for all users
user_user_recommendations = {}

for user in ground_truth.keys():
    user_user_recommendations[user] = recommend_user_user(
        target_user=user,
        train_user_items=train_user_items,
        item_users=item_users,
        top_n_neighbors=10,
        k=10
    )

print("Number of users with recommendations:", len(user_user_recommendations))


Number of users with recommendations: 7838


In [99]:
# Evaluate the user-user collaborative filtering recommender
user_user_results = evaluate_recommender(user_user_recommendations, ground_truth, k=10)

print("User-user collaborative filtering results:")
print(user_user_results)

User-user collaborative filtering results:
{'Precision@K': 0.005651952028578719, 'Recall@K': 0.05651952028578719}


## User-user collaborative filtering results

The user-user collaborative filtering model performs substantially better than the popularity baseline.

This shows that personalization helps: using the behavior of similar users allows the model to recover the true next item for a larger fraction of users than a non-personalized popularity strategy.

Although the scores are still modest, this result confirms that collaborative filtering is more appropriate for this recommendation task than a simple popularity-based approach.

In [100]:
# Compare the popularity baseline and the user-user collaborative filtering model
comparison_df = pd.DataFrame({
    "Model": ["Popularity baseline", "User-user CF"],
    "Precision@10": [
        popular_results["Precision@K"],
        user_user_results["Precision@K"]
    ],
    "Recall@10": [
        popular_results["Recall@K"],
        user_user_results["Recall@K"]
    ]
})

display(comparison_df)

,Model,Precision@10,Recall@10
0,Popularity baseline,0.000408,0.004083
1,User-user CF,0.005652,0.056520


## Comparison of the first two methods

The comparison shows that user-user collaborative filtering performs much better than the popularity baseline.

This means that personalization is important for this task: using the behavior of similar users gives better recommendations than simply suggesting the same globally popular books to everyone.

## Item-item collaborative filtering

We now test an item-item collaborative filtering approach.

Instead of comparing users, this method compares books. The idea is that two books are similar if they are read by many of the same users.

For a target user, we look at the books already seen in training and recommend new books that are similar to them.

In [101]:
# Function to compute similarity between two items using Jaccard similarity
def item_similarity(item_a, item_b, item_users):
    users_a = item_users.get(item_a, set())
    users_b = item_users.get(item_b, set())

    intersection = len(users_a & users_b)
    union = len(users_a | users_b)

    if union == 0:
        return 0

    return intersection / union

In [102]:
# Test the item similarity function on two example items
item_a = 0
item_b = 1

sim_items = item_similarity(item_a, item_b, item_users)

print("Item A:", item_a)
print("Users of item A:", item_users.get(item_a, set()))
print()
print("Item B:", item_b)
print("Users of item B:", item_users.get(item_b, set()))
print()
print("Jaccard similarity between items:", sim_items)

Item A: 0
Users of item A: {0, 224, 6018}

Item B: 1
Users of item B: {0, 2369}

Jaccard similarity between items: 0.25


In [103]:
# Function to recommend items for one user using item-item collaborative filtering
def recommend_item_item(target_user, train_user_items, item_users, k=10):
    # Items already seen by the target user
    seen_items = train_user_items.get(target_user, set())

    # Dictionary to accumulate recommendation scores
    item_scores = {}

    # For each item seen by the user
    for seen_item in seen_items:
        # Compare it to all items in the training item dictionary
        for candidate_item in item_users.keys():
            # Skip items already seen
            if candidate_item in seen_items:
                continue

            # Compute similarity between the seen item and the candidate item
            sim = item_similarity(seen_item, candidate_item, item_users)

            # Accumulate the score
            item_scores[candidate_item] = item_scores.get(candidate_item, 0) + sim

    # Sort items by score in descending order
    ranked_items = sorted(item_scores.items(), key=lambda x: x[1], reverse=True)

    # Keep only the top-k item IDs
    recommended_items = [item for item, score in ranked_items[:k]]

    return recommended_items

## Evaluation of item-item collaborative filtering

We now apply the item-item collaborative filtering method to all users in the validation set and evaluate its performance with Precision@10 and Recall@10.

In [ ]:
# Build item-item recommendations for all users
item_item_recommendations = {}

for user in ground_truth.keys():
    item_item_recommendations[user] = recommend_item_item(
        target_user=user,
        train_user_items=train_user_items,
        item_users=item_users,
        k=10
    )

print("Number of users with recommendations:", len(item_item_recommendations))

In [ ]:
# Evaluate the item-item collaborative filtering recommender
item_item_results = evaluate_recommender(item_item_recommendations, ground_truth, k=10)

print("Item-item collaborative filtering results:")
print(item_item_results)

In [ ]:
# Compare the three methods
comparison_df = pd.DataFrame({
    "Model": ["Popularity baseline", "User-user CF", "Item-item CF"],
    "Precision@10": [
        popular_results["Precision@K"],
        user_user_results["Precision@K"],
        item_item_results["Precision@K"]
    ],
    "Recall@10": [
        popular_results["Recall@K"],
        user_user_results["Recall@K"],
        item_item_results["Recall@K"]
    ]
})

comparison_df["Precision@10"] = comparison_df["Precision@10"].round(6)
comparison_df["Recall@10"] = comparison_df["Recall@10"].round(6)

display(comparison_df)